# GraphNet — First Stack

A 5-minute tour of building and inspecting a heterogeneous Stack architecture
via the GraphNet REPL.

Requires `maturin develop --release` first to build the native extension.


In [ ]:
import graphnet
from graphnet.viz import (
    forward_trace_heatmap,
    hypervector_heatmap,
    similarity_matrix,
    stack_graph,
)

graphnet.banner()

## Build a Stack

A Stack at dimensionality `D=10000` with three heterogeneous operations:
Identity (skip-connection), Dense (HDC bind), HrrBind (FFT-based binding).

Each operation acts on the same input hypervector; the outputs are bundled
into one final hypervector. This is **not** Mixture of Experts — single
bundled output.

In [ ]:
D = 10_000
key_dense = graphnet.Hypervector.random(D, seed=1)
key_hrr = graphnet.Hypervector.random(D, seed=2)

stack = graphnet.Stack(D)
stack.add_operation(graphnet.Operation.identity())
stack.add_operation(graphnet.Operation.dense(key_dense))
stack.add_operation(graphnet.Operation.hrr_bind(key_hrr))
stack

## See the architecture

`stack_graph` renders the architecture as a graphviz diagram.

In [ ]:
stack_graph(stack)

## Run a forward pass with a trace

`forward_with_trace` captures every per-operation output alongside the
bundled result.

In [ ]:
v = graphnet.Hypervector.random(D, seed=42)
trace = stack.forward_with_trace(v)
trace

Visualise the cascade with `forward_trace_heatmap`.

In [ ]:
forward_trace_heatmap(trace);

## Inspect a single Hypervector

`hypervector_heatmap` reshapes the bipolar D-dim vector into a roughly-square
grid for visual scanning.

In [ ]:
hypervector_heatmap(v, width=100);

## Compare a few hypervectors via cosine similarity

In [ ]:
vs = [graphnet.Hypervector.random(D, seed=s) for s in (1, 2, 3, 4, 5)]
similarity_matrix(vs, labels=[f's={s}' for s in (1, 2, 3, 4, 5)]);

## Live intervention

Modify the Stack architecture on the fly and undo it.

In [ ]:
print('before:', stack.op_tags())
token = stack.apply_intervention('add', op=graphnet.Operation.identity())
print('after add:', stack.op_tags())
stack.undo_intervention(token)
print('after undo:', stack.op_tags())